In [1]:
"""
SEED-IV 4-Class Emotion Recognition: Full STMAE + MSC-TimesNet + Marginal UDA Pipeline
=====================================================================================
- Spatial Backbone  : Continuous 2D Scalp-Topology Masked Autoencoder (STMAE)
- Temporal Backbone : Multi-Scale Convolutional TimesNet (MSC-TimesNet) with FFT folding
- Adaptation        : Marginal DANN (GRL on feature latent) + Deep CORAL covariance loss
- Safe Training     : No target entropy minimization or pseudo-labeling (prevents mode collapse)
"""

import os
import math
import time
import copy
import random
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# =====================================================================
# CONFIGURATION & HYPERPARAMETERS
# =====================================================================
DATA_SEARCH_PATHS = [
    "/kaggle/input/datasets/phhasian0710/seed-iv/eeg_feature_smooth",
    "/kaggle/input/seed-iv/eeg_feature_smooth",
    "/kaggle/input/seed-iv-dataset/eeg_feature_smooth",
    "."
]
OUTPUT_DIR = "/kaggle/working/seediv_stmae_timesnet_uda"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_CHANNELS = 62
NUM_CLASSES = 4          # 0: Neutral, 1: Sad, 2: Fear, 3: Happy[cite: 1]
TRIALS_PER_SESSION = 24   #[cite: 1]
RAW_DIM = 10             # 5 log-PSD + 5 DE[cite: 1]
DE_KEY_PREFIX = "de_LDS"   #[cite: 1]
PSD_KEY_PREFIX = "psd_LDS" #[cite: 1]

# Spatial STMAE Parameters
EMBEDDING_SIZE = 32
AUTOENCODER_HIDDEN_SIZE = 64
AUTOENCODER_HEADS = 4
AUTOENCODER_LAYERS = 3
AUTOENCODER_EPOCHS = 20
AUTOENCODER_LR = 1e-3
AUTOENCODER_BATCH_SIZE = 256
RANDOM_MASK_FRACTION = 0.40
REGION_MASK_CHANCE = 0.60

# Temporal MSC-TimesNet Parameters
CLASSIFIER_HIDDEN_SIZE = 128
CLASSIFIER_BLOCKS = 2
TOP_FREQUENCIES = 3
CLASSIFIER_DROPOUT = 0.3

# Sequence Parameters
WINDOW_LENGTH = 10
STRIDE = 1

# Training & UDA Optimization
UDA_EPOCHS = 35
WARMUP_EPOCHS = 5
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
STMAE_FINETUNE_LR = 1e-4
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP = 1.0
LABEL_SMOOTHING = 0.05
CHANNEL_DROPOUT_RATE = 0.1

# Alignment Weights (Marginal DANN + CORAL)
WEIGHT_DOMAIN = 1.0
WEIGHT_CORAL = 5.0

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================================
# ROW-WISE CHANNEL AND TRIAL DEFINITIONS
# =====================================================================
CHANNEL_NAMES = [
    # Row 1 (Pre-Frontal)
    "FP1", "FPZ", "FP2",
    # Row 2 (Anterior-Frontal)
    "AF3", "AF4",
    # Row 3 (Frontal)
    "F7", "F5", "F3", "F1", "FZ", "F2", "F4", "F6", "F8",
    # Row 4 (Fronto-Central)
    "FT7", "FC5", "FC3", "FC1", "FCZ", "FC2", "FC4", "FC6", "FT8",
    # Row 5 (Central / Temporal)
    "T7", "C5", "C3", "C1", "CZ", "C2", "C4", "C6", "T8",
    # Row 6 (Centro-Parietal)
    "TP7", "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6", "TP8",
    # Row 7 (Parietal)
    "P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8",
    # Row 8 (Parieto-Occipital)
    "PO7", "PO5", "PO3", "POZ", "PO4", "PO6", "PO8",
    # Row 9 (Occipital)
    "CB1", "O1", "OZ", "O2", "CB2"
]
assert len(CHANNEL_NAMES) == NUM_CHANNELS

TRIAL_LABELS_BY_SESSION = {
    1: [1, 2, 3, 0, 2, 0, 0, 1, 0, 1, 2, 1, 1, 1, 2, 3, 2, 2, 3, 3, 0, 3, 0, 3],
    2: [2, 1, 3, 0, 0, 2, 0, 2, 3, 3, 2, 3, 2, 0, 1, 1, 2, 1, 0, 3, 0, 1, 3, 1],
    3: [1, 2, 2, 1, 3, 3, 3, 1, 1, 2, 1, 0, 2, 3, 3, 0, 2, 3, 0, 0, 2, 0, 1, 0],
}  #[cite: 1]

REPORT = []


def seed_everything(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =====================================================================
# 2D SCALP TOPOLOGY & REGIONAL PARTITIONS
# =====================================================================
def build_scalp_coords():
    rows = [
        (["FP1", "FPZ", "FP2"], 0.95),
        (["AF3", "AF4"], 0.80),
        (["F7", "F5", "F3", "F1", "FZ", "F2", "F4", "F6", "F8"], 0.62),
        (["FT7", "FC5", "FC3", "FC1", "FCZ", "FC2", "FC4", "FC6", "FT8"], 0.42),
        (["T7", "C5", "C3", "C1", "CZ", "C2", "C4", "C6", "T8"], 0.20),
        (["TP7", "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6", "TP8"], -0.02),
        (["P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8"], -0.25),
        (["PO7", "PO5", "PO3", "POZ", "PO4", "PO6", "PO8"], -0.50),
        (["CB1", "O1", "OZ", "O2", "CB2"], -0.72),
    ]
    coords = {}
    for names, y in rows:
        n = len(names)
        xs = np.linspace(-1.0, 1.0, n) if n > 1 else np.array([0.0])
        for name, x in zip(names, xs):
            coords[name] = (float(x), float(y))
    return np.array([coords[c] for c in CHANNEL_NAMES], dtype=np.float32)


def build_regions():
    regions = defaultdict(list)
    coords = build_scalp_coords()
    for i, _ in enumerate(CHANNEL_NAMES):
        x, y = coords[i]
        if abs(x) > 0.6 and -0.10 < y < 0.50:
            key = "temporal_left" if x < 0 else "temporal_right"
        elif y >= 0.55:
            key = "frontal"
        elif y >= 0.10:
            key = "central"
        elif y >= -0.35:
            key = "parietal"
        else:
            key = "occipital"
        regions[key].append(i)
    return {k: np.array(v, dtype=np.int64) for k, v in regions.items()}


SCALP_COORDS = build_scalp_coords()
REGIONS = build_regions()


# =====================================================================
# DATA EXTRACTION & PREPROCESSING
# =====================================================================
def find_seed_iv_dir():
    for p in DATA_SEARCH_PATHS:
        if os.path.isdir(p):
            return p
    raise FileNotFoundError("Could not locate SEED-IV features directory.")  #[cite: 1]


def load_seed_iv():
    data_dir = find_seed_iv_dir()
    print(f"  [+] Loading SEED-IV archives from: {data_dir}")  #[cite: 1]
    Xs, ys, subs, sess, tris = [], [], [], [], []

    for session in sorted(TRIAL_LABELS_BY_SESSION.keys()):  #[cite: 1]
        sdir = os.path.join(data_dir, str(session))  #[cite: 1]
        if not os.path.isdir(sdir):  #[cite: 1]
            continue  #[cite: 1]
        labels = TRIAL_LABELS_BY_SESSION[session]  #[cite: 1]
        for fname in sorted(os.listdir(sdir)):  #[cite: 1]
            if not fname.endswith(".mat"):  #[cite: 1]
                continue  #[cite: 1]
            subject = int(fname.split("_")[0])  #[cite: 1]
            mat = sio.loadmat(os.path.join(sdir, fname))  #[cite: 1]
            for t in range(1, TRIALS_PER_SESSION + 1):  #[cite: 1]
                dk, pk = f"{DE_KEY_PREFIX}{t}", f"{PSD_KEY_PREFIX}{t}"  #[cite: 1]
                if dk not in mat or pk not in mat:  #[cite: 1]
                    continue  #[cite: 1]
                de = np.transpose(np.asarray(mat[dk], dtype=np.float32), (1, 0, 2))  #[cite: 1]
                psd = np.transpose(np.asarray(mat[pk], dtype=np.float32), (1, 0, 2))  #[cite: 1]
                psd = np.log(np.maximum(psd, 1e-10))  #[cite: 1]
                feat = np.concatenate([psd, de], axis=2)  #[cite: 1]

                n = feat.shape[0]  #[cite: 1]
                Xs.append(feat)  #[cite: 1]
                ys.append(np.full(n, labels[t - 1], dtype=np.int64))  #[cite: 1]
                subs.append(np.full(n, subject, dtype=np.int64))  #[cite: 1]
                sess.append(np.full(n, session, dtype=np.int64))  #[cite: 1]
                tris.append(np.full(n, t, dtype=np.int64))  #[cite: 1]

    X = np.concatenate(Xs, axis=0)  #[cite: 1]
    return (X, np.concatenate(ys), np.concatenate(subs),
            np.concatenate(sess), np.concatenate(tris))  #[cite: 1]


def normalize_subject_session(X, subs, sess):
    Xn = X.copy()  #[cite: 1]
    keys = subs * 100 + sess  #[cite: 1]
    for k in np.unique(keys):  #[cite: 1]
        m = (keys == k)  #[cite: 1]
        blk = Xn[m]  #[cite: 1]
        mu = blk.mean(axis=0, keepdims=True)  #[cite: 1]
        sd = blk.std(axis=0, keepdims=True) + 1e-6  #[cite: 1]
        Xn[m] = (blk - mu) / sd  #[cite: 1]
    return Xn  #[cite: 1]


def build_sequence_index(y, subs, sess, tri, seq_len=WINDOW_LENGTH, stride=STRIDE):
    keys = subs * 1000000 + sess * 10000 + tri  #[cite: 1]
    seqs, labs, s_sub, s_ses, s_tri = [], [], [], [], []  #[cite: 1]
    order = np.argsort(keys, kind="stable")  #[cite: 1]
    for k in np.unique(keys):  #[cite: 1]
        rows = order[keys[order] == k]  #[cite: 1]
        if rows.shape[0] < seq_len:  #[cite: 1]
            continue  #[cite: 1]
        for start in range(0, rows.shape[0] - seq_len + 1, stride):  #[cite: 1]
            win = rows[start:start + seq_len]  #[cite: 1]
            seqs.append(win)  #[cite: 1]
            labs.append(y[win[0]])  #[cite: 1]
            s_sub.append(subs[win[0]])  #[cite: 1]
            s_ses.append(sess[win[0]])  #[cite: 1]
            s_tri.append(tri[win[0]])  #[cite: 1]
    return (np.asarray(seqs, dtype=np.int64), np.asarray(labs, dtype=np.int64),
            np.asarray(s_sub, dtype=np.int64), np.asarray(s_ses, dtype=np.int64),
            np.asarray(s_tri, dtype=np.int64))  #[cite: 1]


# =====================================================================
# SPATIAL STMAE (TOPOLOGY AUTOENCODER)
# =====================================================================
class ScalpPositionalEncoding(nn.Module):
    def __init__(self, coords, d_model):
        super().__init__()
        self.register_buffer("coords", torch.tensor(coords, dtype=torch.float32))
        self.mlp = nn.Sequential(
            nn.Linear(2, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model)
        )

    def forward(self, x):
        return x + self.mlp(self.coords).unsqueeze(0)


class STMAE(nn.Module):
    def __init__(self, in_feat, d_model=AUTOENCODER_HIDDEN_SIZE, latent_dim=EMBEDDING_SIZE,
                 heads=AUTOENCODER_HEADS, layers=AUTOENCODER_LAYERS):
        super().__init__()
        self.proj = nn.Linear(in_feat, d_model)
        self.pos = ScalpPositionalEncoding(SCALP_COORDS, d_model)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.mask_token, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads, dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True, norm_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.to_latent = nn.Linear(d_model, latent_dim)
        self.latent_norm = nn.LayerNorm(latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, in_feat)
        )

    def encode(self, x, mask=None):
        h = self.proj(x)
        if mask is not None:
            h = torch.where(mask.unsqueeze(-1), self.mask_token.expand_as(h), h)
        h = self.pos(h)
        h = self.encoder(h)
        return self.latent_norm(self.to_latent(h))

    def forward(self, x, mask):
        z = self.encode(x, mask)
        return self.decoder(z), z


def sample_mask(batch_size, device):
    mask = torch.zeros(batch_size, NUM_CHANNELS, dtype=torch.bool, device=device)
    region_keys = list(REGIONS.keys())
    for b in range(batch_size):
        if random.random() < REGION_MASK_CHANCE:
            k = random.choice([1, 2])
            for key in random.sample(region_keys, k):
                mask[b, torch.tensor(REGIONS[key], device=device)] = True
        else:
            n = max(1, int(RANDOM_MASK_FRACTION * NUM_CHANNELS))
            idx = torch.randperm(NUM_CHANNELS, device=device)[:n]
            mask[b, idx] = True
    return mask


def pretrain_stmae(X_pt):
    in_feat = X_pt.shape[2]
    model = STMAE(in_feat).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=AUTOENCODER_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=AUTOENCODER_EPOCHS)
    n = X_pt.shape[0]
    model.train()
    for ep in range(1, AUTOENCODER_EPOCHS + 1):
        perm = torch.randperm(n)
        tot, nb = 0.0, 0
        for i in range(0, n, AUTOENCODER_BATCH_SIZE):
            xb = X_pt[perm[i:i + AUTOENCODER_BATCH_SIZE]]
            mask = sample_mask(xb.shape[0], DEVICE)
            recon, _ = model(xb, mask)
            m = mask.unsqueeze(-1).expand_as(xb)
            loss = F.mse_loss(recon[m], xb[m])
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            opt.step()
            tot += loss.item()
            nb += 1
        sched.step()
        if ep == 1 or ep % 5 == 0:
            print("  STMAE ep%03d masked_recon_mse=%.5f" % (ep, tot / max(nb, 1)))
    return model


# =====================================================================
# TEMPORAL MSC-TIMESNET MODULES
# =====================================================================
def fft_topk_periods(x, k=TOP_FREQUENCIES):
    B, T, d = x.shape
    xf = torch.fft.rfft(x, dim=1)
    amp = xf.abs().mean(dim=2)
    amp[:, 0] = 0.0
    k = min(k, max(amp.shape[1] - 1, 1))
    _, idx = torch.topk(amp, k, dim=1)
    freqs = idx.float().mean(dim=0).round().long().clamp(min=1)
    periods = [max(int(T // f.item()), 1) for f in freqs]
    weights = torch.stack([amp[:, i] for i in freqs], dim=1)
    return periods, F.softmax(weights, dim=1)


class MultiScaleConvBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        h = max(d_model // 4, 8)
        self.b1 = nn.Sequential(
            nn.Conv2d(d_model, h, 1),
            nn.BatchNorm2d(h),
            nn.GELU()
        )
        self.b3 = nn.Sequential(
            nn.Conv2d(d_model, h, 3, padding=1),
            nn.BatchNorm2d(h),
            nn.GELU()
        )
        self.b5 = nn.Sequential(
            nn.Conv2d(d_model, h, 5, padding=2),
            nn.BatchNorm2d(h),
            nn.GELU()
        )
        self.bp = nn.Sequential(
            nn.AvgPool2d(3, stride=1, padding=1),
            nn.Conv2d(d_model, h, 1),
            nn.BatchNorm2d(h),
            nn.GELU()
        )
        self.fuse = nn.Sequential(
            nn.Conv2d(4 * h, d_model, 1),
            nn.BatchNorm2d(d_model)
        )

    def forward(self, x):
        return self.fuse(torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1))


class TimesBlock(nn.Module):
    def __init__(self, d_model, topk=TOP_FREQUENCIES):
        super().__init__()
        self.topk = topk
        self.conv = MultiScaleConvBlock(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        B, T, d = x.shape
        periods, weights = fft_topk_periods(x, self.topk)
        outs = []
        for p in periods:
            pad = (math.ceil(T / p) * p) - T
            xp = F.pad(x, (0, 0, 0, pad)) if pad > 0 else x
            Tp = xp.shape[1]
            num_p = Tp // p
            z = xp.permute(0, 2, 1).reshape(B, d, num_p, p)
            z = self.conv(z)
            z = z.reshape(B, d, Tp).permute(0, 2, 1)[:, :T, :]
            outs.append(z)
        stacked = torch.stack(outs, dim=-1)
        w = weights.unsqueeze(1).unsqueeze(1)
        agg = (stacked * w).sum(dim=-1)
        return self.norm(agg + x)


# =====================================================================
# UDA: GRADIENT REVERSAL & CORAL
# =====================================================================
class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None


def grad_reverse(x, alpha=1.0):
    return GradReverse.apply(x, alpha)


def coral_loss(source, target):
    d = source.size(1)
    ns, nt = source.size(0), target.size(0)
    if ns <= 1 or nt <= 1:
        return torch.tensor(0.0, device=source.device)

    ones_s = torch.ones(ns, 1, device=source.device)
    hs = torch.eye(ns, device=source.device) - (1.0 / ns) * torch.mm(ones_s, ones_s.t())
    cs = (1.0 / (ns - 1)) * torch.mm(source.t(), torch.mm(hs, source))

    ones_t = torch.ones(nt, 1, device=target.device)
    ht = torch.eye(nt, device=target.device) - (1.0 / nt) * torch.mm(ones_t, ones_t.t())
    ct = (1.0 / (nt - 1)) * torch.mm(target.t(), torch.mm(ht, target))

    return torch.sum((cs - ct) ** 2) / (4.0 * (d * d))


# =====================================================================
# FULL INTEGRATED ARCHITECTURE
# =====================================================================
class FullProposedUDANetwork(nn.Module):
    def __init__(self, stmae, num_classes=NUM_CLASSES, hidden_dim=CLASSIFIER_HIDDEN_SIZE, blocks=CLASSIFIER_BLOCKS):
        super().__init__()
        self.stmae = stmae
        in_dim = NUM_CHANNELS * EMBEDDING_SIZE

        self.inp_proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )
        self.times_blocks = nn.ModuleList([
            TimesBlock(hidden_dim) for _ in range(blocks)
        ])
        self.feat_norm = nn.LayerNorm(hidden_dim)

        self.class_classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(CLASSIFIER_DROPOUT),
            nn.Linear(64, num_classes)
        )

        self.domain_discriminator = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(CLASSIFIER_DROPOUT),
            nn.Linear(64, 1)
        )

    def extract_features(self, x):
        B, T, C, Fq = x.shape
        flat = x.reshape(B * T, C, Fq)
        z = self.stmae.encode(flat).reshape(B, T, C * EMBEDDING_SIZE)
        h = self.inp_proj(z)
        for blk in self.times_blocks:
            h = blk(h)
        return self.feat_norm(h.mean(dim=1))

    def forward(self, x, alpha=1.0):
        feat = self.extract_features(x)
        class_logits = self.class_classifier(feat)
        domain_logits = self.domain_discriminator(grad_reverse(feat, alpha)).squeeze(-1)
        return class_logits, domain_logits, feat


# =====================================================================
# TRAINING & EVALUATION FUNCTIONS
# =====================================================================
def get_lr(epoch, total_epochs=UDA_EPOCHS, warmup=WARMUP_EPOCHS, base_lr=LEARNING_RATE, min_lr=1e-5):
    if epoch <= warmup:
        return base_lr * epoch / warmup
    progress = (epoch - warmup) / max(1, total_epochs - warmup)
    return min_lr + 0.5 * (base_lr - min_lr) * (1.0 + math.cos(math.pi * progress))


def apply_channel_dropout(xb, p=CHANNEL_DROPOUT_RATE):
    if p <= 0:
        return xb
    B = xb.shape[0]
    keep = (torch.rand(B, 1, NUM_CHANNELS, 1, device=xb.device) > p).float()
    return xb * keep


def trial_level_scores(probs, y, sub, ses, tri):
    keys = sub * 1000000 + ses * 10000 + tri
    yt, yp = [], []
    for k in np.unique(keys):
        m = (keys == k)
        yt.append(np.bincount(y[m]).argmax())
        yp.append(probs[m].mean(axis=0).argmax())

    yt, yp = np.array(yt), np.array(yp)
    return (accuracy_score(yt, yp),
            balanced_accuracy_score(yt, yp),
            f1_score(yt, yp, average="macro", zero_division=0),
            f1_score(yt, yp, average="weighted", zero_division=0),
            len(yt))


def run_full_uda_fold(target_sub, pretrained_stmae, X_pt, seq_idx_pt, packs):
    labels, s_sub, s_ses, s_tri = packs[1:]

    source_idx = np.where(s_sub != target_sub)[0]
    target_idx = np.where(s_sub == target_sub)[0]

    num_source = len(source_idx)
    num_target = len(target_idx)
    steps_per_epoch = max(num_source // BATCH_SIZE, 1)

    seed_everything(RANDOM_SEED)

    stmae_fold = copy.deepcopy(pretrained_stmae)
    for p in stmae_fold.parameters():
        p.requires_grad = True

    model = FullProposedUDANetwork(stmae_fold).to(DEVICE)

    # Differential Learning Rates: fine-tune STMAE gently to preserve 2D topology
    optimizer = torch.optim.AdamW([
        {"params": model.stmae.parameters(), "lr": STMAE_FINETUNE_LR, "weight_decay": 1e-4},
        {"params": model.inp_proj.parameters(), "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
        {"params": model.times_blocks.parameters(), "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
        {"params": model.feat_norm.parameters(), "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
        {"params": model.class_classifier.parameters(), "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
        {"params": model.domain_discriminator.parameters(), "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
    ])

    crit_class = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    crit_domain = nn.BCEWithLogitsLoss()

    for epoch in range(1, UDA_EPOCHS + 1):
        lr_main = get_lr(epoch, base_lr=LEARNING_RATE)
        lr_stmae = get_lr(epoch, base_lr=STMAE_FINETUNE_LR)

        optimizer.param_groups[0]["lr"] = lr_stmae
        for g in optimizer.param_groups[1:]:
            g["lr"] = lr_main

        model.train()
        src_perm = np.random.permutation(source_idx)
        tgt_perm = np.random.permutation(target_idx)

        for step in range(steps_per_epoch):
            p = (step + (epoch - 1) * steps_per_epoch) / (UDA_EPOCHS * steps_per_epoch)
            alpha = float(2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0)

            # 1. Source Batch (With Labels)
            s_batch_rows = src_perm[(step * BATCH_SIZE) % (num_source - BATCH_SIZE):
                                    (step * BATCH_SIZE) % (num_source - BATCH_SIZE) + BATCH_SIZE]
            xb_s = apply_channel_dropout(X_pt[seq_idx_pt[s_batch_rows]])
            yb_s = torch.tensor(labels[s_batch_rows], dtype=torch.long, device=DEVICE)

            # 2. Target Batch (Zero Labels Accessed)
            t_batch_rows = tgt_perm[(step * BATCH_SIZE) % (num_target - BATCH_SIZE):
                                    (step * BATCH_SIZE) % (num_target - BATCH_SIZE) + BATCH_SIZE]
            xb_t = apply_channel_dropout(X_pt[seq_idx_pt[t_batch_rows]])

            # Forward Passes
            cls_s, dom_s, feat_s = model(xb_s, alpha=alpha)
            _, dom_t, feat_t = model(xb_t, alpha=alpha)

            # Marginal Loss Formulation
            loss_cls = crit_class(cls_s, yb_s)
            loss_dom = 0.5 * (crit_domain(dom_s, torch.zeros_like(dom_s)) +
                              crit_domain(dom_t, torch.ones_like(dom_t)))
            loss_coral = coral_loss(feat_s, feat_t)

            total_loss = loss_cls + (WEIGHT_DOMAIN * loss_dom) + (WEIGHT_CORAL * loss_coral)

            optimizer.zero_grad()
            total_loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()

    # =====================================================================
    # TEST INFERENCE
    # =====================================================================
    model.eval()
    all_probs = []
    with torch.no_grad():
        for i in range(0, num_target, BATCH_SIZE):
            b_rows = target_idx[i:i + BATCH_SIZE]
            xb = X_pt[seq_idx_pt[b_rows]]
            logits, _, _ = model(xb, alpha=0.0)
            all_probs.append(F.softmax(logits, dim=1).cpu().numpy())

    probs = np.concatenate(all_probs, axis=0)
    yt = labels[target_idx]
    pred_win = probs.argmax(axis=1)

    win_acc = accuracy_score(yt, pred_win)
    win_bacc = balanced_accuracy_score(yt, pred_win)
    win_macro_f1 = f1_score(yt, pred_win, average="macro", zero_division=0)
    win_weighted_f1 = f1_score(yt, pred_win, average="weighted", zero_division=0)

    t_acc, t_bacc, t_macro_f1, t_weighted_f1, num_trials = trial_level_scores(
        probs, yt, s_sub[target_idx], s_ses[target_idx], s_tri[target_idx]
    )

    fold_name = f"subject_{target_sub:02d}"
    print(
        f"  -> {fold_name:<11} | "
        f"WIN: acc={win_acc:.4f} bacc={win_bacc:.4f} macF1={win_macro_f1:.4f} wF1={win_weighted_f1:.4f} | "
        f"TRIAL: acc={t_acc:.4f} bacc={t_bacc:.4f} macF1={t_macro_f1:.4f} wF1={t_weighted_f1:.4f} "
        f"({num_target}w / {num_trials}t)"
    )

    REPORT.append(dict(
        fold=fold_name,
        win_acc=win_acc, win_bacc=win_bacc, win_macro_f1=win_macro_f1, win_weighted_f1=win_weighted_f1,
        trial_acc=t_acc, trial_bacc=t_bacc, trial_macro_f1=t_macro_f1, trial_weighted_f1=t_weighted_f1,
        num_windows=num_target, num_trials=num_trials
    ))


# =====================================================================
# MAIN PIPELINE
# =====================================================================
def main():
    t0 = time.time()
    seed_everything(RANDOM_SEED)
    print("device:", DEVICE)
    print("Task: SEED-IV 4-Class LOSO Evaluation | Mode: FULL ARCHITECTURE (STMAE + MSC-TIMESNET + UDA)")

    print("\n[1] Loading data from .mat archives...")
    X_raw, y, subs, sess, tri = load_seed_iv()
    print(f"shape={X_raw.shape}  labels={np.bincount(y)}")  #[cite: 1]
    print("subjects:", sorted(np.unique(subs).tolist()))  #[cite: 1]
    print("sessions:", sorted(np.unique(sess).tolist()))  #[cite: 1]

    print("\n[2] Normalizing (Per-Subject-Session Standardization)...")
    X = normalize_subject_session(X_raw, subs, sess)

    print("\n[2.5] Pushing normalized dataset to GPU VRAM...")
    X_pt = torch.tensor(X, dtype=torch.float32, device=DEVICE)

    print("\n[3] Stage A: Pretraining STMAE on 62-channel 2D scalp coordinates...")
    pretrained_stmae = pretrain_stmae(X_pt)

    print("\n[4] Building sliding windows (T=10, Stride=1)...")
    packs = build_sequence_index(y, subs, sess, tri, WINDOW_LENGTH, stride=STRIDE)
    seq_idx = packs[0]
    seq_idx_pt = torch.tensor(seq_idx, dtype=torch.long, device=DEVICE)
    print(f"  {seq_idx.shape[0]} windows constructed. Labels distribution: {np.bincount(packs[1])}")  #[cite: 1]

    # -----------------------------------------------------------------
    # 15-FOLD LOSO LOOP
    # -----------------------------------------------------------------
    print("\n" + "=" * 84)
    print("EVALUATION: SEED-IV 4-CLASS 15-FOLD LOSO (STMAE + MSC-TIMESNET + UDA)")
    print("=" * 84)

    s_sub = packs[2]
    unique_subs = sorted(np.unique(s_sub).tolist())

    for target_sub in unique_subs:
        run_full_uda_fold(target_sub, pretrained_stmae, X_pt, seq_idx_pt, packs)

    df = pd.DataFrame(REPORT)
    results_path = os.path.join(OUTPUT_DIR, "results_seediv_full_stmae_timesnet_uda.csv")
    df.to_csv(results_path, index=False)
    print(f"\nsaved: {results_path}")

    print("\n" + "=" * 84)
    print("SUMMARY -- SEED-IV 4-CLASS LOSO (FULL STMAE + MSC-TIMESNET + DANN/CORAL)")
    print("=" * 84)
    if not df.empty:
        summary_dict = {
            "win_acc": [df["win_acc"].mean()],
            "win_bacc": [df["win_bacc"].mean()],
            "win_macro_f1": [df["win_macro_f1"].mean()],
            "win_weighted_f1": [df["win_weighted_f1"].mean()],
            "trial_acc": [df["trial_acc"].mean()],
            "trial_bacc": [df["trial_bacc"].mean()],
            "trial_macro_f1": [df["trial_macro_f1"].mean()],
            "trial_weighted_f1": [df["trial_weighted_f1"].mean()],
            "folds": [len(df)]
        }
        summ = pd.DataFrame(summary_dict)
        print(summ.to_string(index=False, float_format=lambda v: "%.4f" % v))

    print(f"\nTotal Wall time: {(time.time() - t0) / 60.0:.1f} min")


if __name__ == "__main__":
    main()

device: cuda
Task: SEED-IV 4-Class LOSO Evaluation | Mode: FULL ARCHITECTURE (STMAE + MSC-TIMESNET + UDA)

[1] Loading data from .mat archives...
  [+] Loading SEED-IV archives from: /kaggle/input/datasets/phhasian0710/seed-iv/eeg_feature_smooth
shape=(37575, 62, 10)  labels=[10170 10245  9225  7935]
subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
sessions: [1, 2, 3]

[2] Normalizing (Per-Subject-Session Standardization)...

[2.5] Pushing normalized dataset to GPU VRAM...

[3] Stage A: Pretraining STMAE on 62-channel 2D scalp coordinates...
  STMAE ep001 masked_recon_mse=0.44195
  STMAE ep005 masked_recon_mse=0.22124
  STMAE ep010 masked_recon_mse=0.19666
  STMAE ep015 masked_recon_mse=0.18780
  STMAE ep020 masked_recon_mse=0.18491

[4] Building sliding windows (T=10, Stride=1)...
  27855 windows constructed. Labels distribution: [7740 7815 6795 5505]

EVALUATION: SEED-IV 4-CLASS 15-FOLD LOSO (STMAE + MSC-TIMESNET + UDA)
  -> subject_01  | WIN: acc=0.7447 bacc=0.7355 macF